Experiment to check how the attack performs on noisy points and structural points

In [8]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time
import json
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# DATA_DIR = Path('../../data/spambase_50_50')
DATA_DIR = Path('/home/skada009/aro/spambase_10_40_50')
SCORES_DIR = Path('/home/skada009/aro/experiments/noise_scoring_outputs/20260324_203954')
# OUTPUT_DIR = Path('../../outputs/targeted_vaccination_50_50')
OUTPUT_DIR = Path('/home/skada009/aro/targeted_vaccination_10_40_50')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.1
MAXITERS = 300
LAMBDAS = {'aggressive': 0.1, 'stealthy': 50.0}
ATTACK_PCTS = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
K_RANKS = [1, 2, 3, 5, 10, 15, 20, 30, 40, 50, 57]

Device: cuda
GPU: NVIDIA RTX A6000


In [9]:
data = np.load(DATA_DIR / 'train_test_data.npz')
X_test = data['X_test'].astype(np.float32)
y_test = data['y_test']
n_test, n_features = X_test.shape

prep = np.load(DATA_DIR / 'preprocessing.npz')
weights = prep['weights']
bounds = [prep['bounds_min'], prep['bounds_max']]


class SpambaseNet(nn.Module):
    def __init__(self, D_in):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(D_in, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 2), nn.Softmax(dim=-1)
        )

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
            return self.layer(x).squeeze(0)
        return self.layer(x)


checkpoint = torch.load(DATA_DIR / 'spambase_mlp.pth', map_location=device)
model = SpambaseNet(checkpoint['D_in']).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

X_test_t = torch.FloatTensor(X_test).to(device)
with torch.no_grad():
    test_preds = model(X_test_t).argmax(dim=1).cpu().numpy()
correct_mask = (test_preds == y_test)
correct_indices = np.where(correct_mask)[0]
n_correct = len(correct_indices)

print(f'Test set: {n_test} samples, {n_features} features')
print(f'Correctly classified: {n_correct}/{n_test} ({n_correct/n_test:.2%})')

Test set: 2301 samples, 57 features
Correctly classified: 2058/2301 (89.44%)


In [10]:
score1_removal_order = np.load(SCORES_DIR / 'test_clean_data_score1_removed_index.npy').astype(int)
score2_value = np.load(SCORES_DIR / 'test_clean_data_score2_noise_scores.npy')

# Noisiest-first orderings, filtered to correctly-classified only
score1_order = [idx for idx in score1_removal_order if correct_mask[idx]]
score2_order = [idx for idx in np.argsort(score2_value)[::-1] if correct_mask[idx]]

score_orders = {'Score 1': score1_order, 'Score 2': score2_order}
print(f'Eligible points per score ordering: {len(score1_order)}')

Eligible points per score ordering: 2057


In [11]:
def clip_tensor(current, low_bound, up_bound, dev):
    low_bound = torch.FloatTensor(low_bound).to(dev)
    up_bound = torch.FloatTensor(up_bound).to(dev)
    return torch.max(torch.min(current, up_bound), low_bound)


def lowProFool_attack(x, model, weights, bounds, maxiters, alpha, lambda_, device):
    """Run LowProFool on a single sample. Returns (x_adv_numpy, success_flag)."""
    x = x.to(device)
    w = torch.FloatTensor(np.array(weights)).to(device)
    r = torch.FloatTensor(1e-4 * np.ones(x.shape)).to(device)
    r.requires_grad = True

    with torch.no_grad():
        orig_pred = model(x).argmax().cpu().item()

    target_pred = 1 - orig_pred
    target = torch.tensor([0., 1.] if target_pred == 1 else [1., 0.]).to(device)
    bce = nn.BCELoss()

    for _ in range(maxiters):
        if r.grad is not None:
            r.grad.zero_()
        output = model(x + r)
        loss = bce(output, target) + lambda_ * torch.sqrt(torch.sum((w * r) ** 2))
        loss.backward(retain_graph=True)
        with torch.no_grad():
            r_new = r - alpha * r.grad
        r = r_new.clone().detach().requires_grad_(True)

    x_adv = clip_tensor(x + r, bounds[0], bounds[1], device)
    with torch.no_grad():
        adv_pred = model(x_adv).argmax().cpu().item()

    return x_adv.detach().cpu().numpy().flatten(), int(orig_pred != adv_pred)

In [12]:

rows = []
t_total = time.time()

for score_name, order in score_orders.items():
    n_eligible = len(order)
    print(n_eligible)
    n_attack = max(1, int(n_eligible * 20 / 100))
    print(f'Attacking {n_attack} samples ({n_attack/n_eligible:.2%}) for score: {score_name}')

    # Noisy tier: first n_attack in noisiest-first order
    noisy_indices = order[:n_attack]
    # Structural tier: last n_attack in noisiest-first order (most structural)
    structural_indices = order[-n_attack:]
    print("Noisy indices:", len(noisy_indices))
    print("Structural indices:", len(structural_indices))


2057
Attacking 411 samples (19.98%) for score: Score 1
Noisy indices: 411
Structural indices: 411
2058
Attacking 411 samples (19.97%) for score: Score 2
Noisy indices: 411
Structural indices: 411


In [16]:
# Run LPF attacks on 20% noisy and structural samples from both scores
attack_results = []
rows = []

for score_name, order in score_orders.items():
    n_eligible = len(order)
    n_attack = max(1, int(n_eligible * 20 / 100))
    
    noisy_indices = order[:n_attack]
    structural_indices = order[-n_attack:]
    
    # For each tier, attack with both lambda types
    for tier_name, tier_indices in [('noisy', noisy_indices), ('structural', structural_indices)]:
        for lambda_type, lambda_val in [('aggressive', LAMBDAS['aggressive']), ('stealthy', LAMBDAS['stealthy'])]:
            tier_successes = 0
            for idx in tier_indices:
                x = torch.FloatTensor(X_test[idx]).to(device)
                x_adv, success = lowProFool_attack(x, model, weights, bounds, MAXITERS, ALPHA, lambda_val, device)
                if success:
                    tier_successes += 1
                attack_results.append({
                    'score': score_name,
                    'tier': tier_name,
                    'lambda_type': lambda_type,
                    'lambda_val': lambda_val,
                    'idx': idx,
                    'success': success
                })
            
            # Log aggregated results
            rows.append({
                'score': score_name,
                'tier': tier_name,
                'attack': lambda_type,
                'lambda': lambda_val,
                'n_samples': len(tier_indices),
                'n_successful': tier_successes,
                'success_rate': tier_successes / len(tier_indices) if len(tier_indices) > 0 else 0
            })
            
            print(f"{score_name} - {tier_name.capitalize()} ({lambda_type} λ={lambda_val}): {tier_successes}/{len(tier_indices)} ({tier_successes/len(tier_indices):.2%})")

# Display aggregated results
results_df = pd.DataFrame(rows)
print("\n" + "="*80)
print("=== LowProFool Attack Results - All Combinations ===")
print("="*80)
print(results_df.to_string(index=False))

# Show detailed breakdown by score
print("\n" + "="*80)
print("=== Detailed Breakdown by Score ===")
print("="*80)
detailed_df = pd.DataFrame(attack_results)

for score in detailed_df['score'].unique():
    print(f"\n{score}:")
    score_data = detailed_df[detailed_df['score'] == score]
    for tier in ['noisy', 'structural']:
        print(f"  {tier.capitalize()} Points:")
        tier_data = score_data[score_data['tier'] == tier]
        for lambda_type in ['aggressive', 'stealthy']:
            subset = tier_data[tier_data['lambda_type'] == lambda_type]
            if len(subset) > 0:
                successes = subset['success'].sum()
                lambda_val = subset['lambda_val'].iloc[0]
                print(f"    {lambda_type.capitalize()} (λ={lambda_val}): {successes}/{len(subset)} ({successes/len(subset):.2%})")


Score 1 - Noisy (aggressive λ=0.1): 411/411 (100.00%)
Score 1 - Noisy (stealthy λ=50.0): 344/411 (83.70%)
Score 1 - Structural (aggressive λ=0.1): 411/411 (100.00%)
Score 1 - Structural (stealthy λ=50.0): 173/411 (42.09%)
Score 2 - Noisy (aggressive λ=0.1): 411/411 (100.00%)
Score 2 - Noisy (stealthy λ=50.0): 338/411 (82.24%)
Score 2 - Structural (aggressive λ=0.1): 411/411 (100.00%)
Score 2 - Structural (stealthy λ=50.0): 229/411 (55.72%)

=== LowProFool Attack Results - All Combinations ===
  score       tier     attack  lambda  n_samples  n_successful  success_rate
Score 1      noisy aggressive     0.1        411           411      1.000000
Score 1      noisy   stealthy    50.0        411           344      0.836983
Score 1 structural aggressive     0.1        411           411      1.000000
Score 1 structural   stealthy    50.0        411           173      0.420925
Score 2      noisy aggressive     0.1        411           411      1.000000
Score 2      noisy   stealthy    50.0   

In [18]:
from Adverse import zero_gradients


def clip(current, low_bound, up_bound, device):
    low_bound = torch.FloatTensor(low_bound).to(device)
    up_bound = torch.FloatTensor(up_bound).to(device)
    return torch.max(torch.min(current, up_bound), low_bound)


def deepfool(x_old, net, maxiters, alpha, bounds, weights=[], overshoot=0.002):
    """
    :param image: tabular sample
    :param net: network 
    :param maxiters: maximum number of iterations ran to generate the adversarial examples
    :param alpha: scaling factor used to control the growth of the perturbation
    :param bounds: bounds of the datasets with respect to each feature
    :param weights: feature importance vector associated with the dataset at hand
    :param overshoot: used as a termination criterion to prevent vanishing updates (default = 0.02).
    :return: minimal perturbation that fools the classifier, number of iterations that it required, new estimated_label and perturbed image
    """    
    input_shape = x_old.detach().cpu().numpy().shape
    x = x_old.detach().clone().requires_grad_(True)
    
    output = net.forward(x)
    orig_pred = output.max(0, keepdim=True)[1]  # get the index of the max log-probability
    origin = orig_pred.clone().detach()
    I = []
    if orig_pred.item() == 0:
        I = [0, 1]
    else:
        I = [1, 0]       
    w = np.zeros(input_shape)
    r_tot = np.zeros(input_shape)    
    k_i = origin.clone()
    loop_i = 0
    while torch.eq(k_i, origin) and loop_i < maxiters:                
        # Origin class
        output[I[0]].backward(retain_graph=True)
        grad_orig = x.grad.detach().cpu().numpy().copy()        
        # Target class
        zero_gradients(x)
        output[I[1]].backward(retain_graph=True)
        cur_grad = x.grad.detach().cpu().numpy().copy()            
        # set new w and new f
        w = cur_grad - grad_orig
        f = (output[I[1]] - output[I[0]]).detach().cpu().numpy()
        pert = abs(f)/np.linalg.norm(w.flatten())    
        # compute r_i and r_tot
        # Added 1e-4 for numerical stability
        r_i =  (pert+1e-4) * w / np.linalg.norm(w)          
        if len(weights) > 0:
            r_i /= np.array(weights)
        # limit huge step
        r_i = alpha * r_i / np.linalg.norm(r_i)            
        r_tot = np.float32(r_tot + r_i)       
        pert_x = x_old + (1 + overshoot) * torch.from_numpy(r_tot).to(x_old.device)
        if len(bounds) > 0:
            pert_x = clip(pert_x, bounds[0], bounds[1], x_old.device)               
        x = pert_x.detach().clone().requires_grad_(True)
        output = net.forward(x)      
        k_i = torch.tensor(np.argmax(output.detach().cpu().numpy().flatten()), device=origin.device)                   
        loop_i += 1
    r_tot = (1+overshoot)*r_tot    
    pert_x = clip(pert_x, bounds[0], bounds[1], x_old.device)
    return int(orig_pred.item()), int(k_i.item()), pert_x.detach().cpu().numpy(), loop_i

In [19]:
# Run DF attacks on 20% noisy and structural samples from both scores
attack_results_df = []
rows_df = []
ALPHA = 0.001
for score_name, order in score_orders.items():
    n_eligible = len(order)
    n_attack = max(1, int(n_eligible * 20 / 100))
    
    noisy_indices = order[:n_attack]
    structural_indices = order[-n_attack:]
    
    for tier_name, tier_indices in [('noisy', noisy_indices), ('structural', structural_indices)]:
        tier_successes = 0
        for idx in tier_indices:
            x = torch.FloatTensor(X_test[idx]).to(device)
            orig_pred, adv_pred, x_adv_np, _ = deepfool(x, model, MAXITERS, ALPHA, bounds, weights=[])
            success = 1 if orig_pred != adv_pred else 0
            if success:
                tier_successes += 1
            attack_results_df.append({
                'score': score_name,
                'tier': tier_name,
                'idx': idx,
                'success': success
            })
        
        # Log aggregated results
        rows_df.append({
            'score': score_name,
            'tier': tier_name,
            'n_samples': len(tier_indices),
            'n_successful': tier_successes,
            'success_rate': tier_successes / len(tier_indices) if len(tier_indices) > 0 else 0
        })
        
        print(f"{score_name} - {tier_name.capitalize()}: {tier_successes}/{len(tier_indices)} ({tier_successes/len(tier_indices):.2%})")

# Display aggregated results
results_df_df = pd.DataFrame(rows_df)
print("\n" + "="*80)
print("=== DeepFool Attack Results Summary ===")
print("="*80)
print(results_df_df.to_string(index=False))

# Show detailed breakdown by score
print("\n" + "="*80)
print("=== Detailed Breakdown by Score ===")
print("="*80)
detailed_df_df = pd.DataFrame(attack_results_df)
print(f"Total successful attacks: {detailed_df_df['success'].sum()}/{len(detailed_df_df)} ({detailed_df_df['success'].sum()/len(detailed_df_df):.2%})")

for score in detailed_df_df['score'].unique():
    print(f"\n{score}:")
    score_data = detailed_df_df[detailed_df_df['score'] == score]
    for tier in ['noisy', 'structural']:
        tier_data = score_data[score_data['tier'] == tier]
        successes = tier_data['success'].sum()
        if len(tier_data) > 0:
            print(f"  {tier.capitalize()}: {successes}/{len(tier_data)} ({successes/len(tier_data):.2%})")


Score 1 - Noisy: 307/411 (74.70%)
Score 1 - Structural: 411/411 (100.00%)
Score 2 - Noisy: 327/411 (79.56%)
Score 2 - Structural: 411/411 (100.00%)

=== DeepFool Attack Results Summary ===
  score       tier  n_samples  n_successful  success_rate
Score 1      noisy        411           307      0.746959
Score 1 structural        411           411      1.000000
Score 2      noisy        411           327      0.795620
Score 2 structural        411           411      1.000000

=== Detailed Breakdown by Score ===
Total successful attacks: 1456/1644 (88.56%)

Score 1:
  Noisy: 307/411 (74.70%)
  Structural: 411/411 (100.00%)

Score 2:
  Noisy: 327/411 (79.56%)
  Structural: 411/411 (100.00%)
